In [1]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from simple_pe.param_est import find_metric_and_eigendirections, calc_dx_bounds, pe
from simple_pe.waveforms import waveform
from pesummary.utils.array import Array
from pesummary.utils.samples_dict import SamplesDict

/home/ben.patterson/.conda/envs/igwn_eccentric_new/lib/python3.10/site-packages/pycbc/types/array.py:36: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal as _lal
/home/ben.patterson/.conda/envs/igwn_eccentric_new/lib/python3.10/site-packages/pykerr/qnm.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this p

lal.MSUN_SI != Msun


In [1]:
from simple_pe import io
f_low = 20
asd_data = {'H1': '/home/ben.patterson/projects/simple-pe/examples/zero-noise/aligo_O4high.txt',
            'L1': '/home/ben.patterson/projects/simple-pe/examples/zero-noise/aligo_O4high.txt',
            'V1': '/home/ben.patterson/projects/simple-pe/examples/zero-noise/avirgo_O4high_NEW.txt'}
psds = io.load_psd_from_file(
           {}, asd_data, 1/32, f_low, 2048,
       )
hm_psd = io.calculate_harmonic_mean_psd(psds)

/home/ben.patterson/.conda/envs/igwn_eccentric_new/lib/python3.10/site-packages/pycbc/types/array.py:36: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal as _lal
/home/ben.patterson/.conda/envs/igwn_eccentric_new/lib/python3.10/site-packages/pykerr/qnm.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this p

lal.MSUN_SI != Msun


/home/ben.patterson/.conda/envs/igwn_eccentric_new/lib/python3.10/site-packages/pycbc/types/array.py:390: RuntimeWarning: divide by zero encountered in divide
  return self._data.__rtruediv__(other)


In [39]:
0.9889 + 0.1393**2

1.00830449

In [35]:
snr = 21.2484

x = {'chirp_mass': 53.7231, 'symmetric_mass_ratio': 0.2258, 'chi_align': 0.1890, 'chi_p2': 0.9793}
dx_directions = list(x.keys())

g = find_metric_and_eigendirections(
    x, dx_directions, snr, f_low, hm_psd, approximant="IMRPhenomXPHM",
    mins=None, maxs=None, tolerance=0.05, max_iter=2,
    ncpus=1, mute_multiprocessing=True,
    n_ecc_gen=6, mismatch=None
)

Calculating the metric | iteration 0 < 2| error 0.016 > 0.00086

chi_p2 values smaller than the lower bound, smallest = 1e-06
setting equal to lower bound of 1e-06
chi_p2 values smaller than the lower bound, smallest = 1e-06
setting equal to lower bound of 1e-06


Calculating the metric | iteration 1 < 2| error 0.0059 > 0.00086: 

chi_p2 values smaller than the lower bound, smallest = 1e-06
setting equal to lower bound of 1e-06
chi_p2 values smaller than the lower bound, smallest = 1e-06
setting equal to lower bound of 1e-06
chi_p2 values smaller than the lower bound, smallest = 1e-06
setting equal to lower bound of 1e-06
chi_p2 values smaller than the lower bound, smallest = 1e-06
setting equal to lower bound of 1e-06


Failed to achieve requested tolerance.  Requested: 0.00086achieved 0.0021: 


In [36]:
def create_metric_vectors(
    g, mass_ratio_last=True, eccentricity_last=True
):
    """
    Calculates orthogonal vectors of metric dx space.

    :param g: metric object
    :param mass_ratio_last: whether to prioritise mass ratio parameter last
    :param eccentricity_last: whether to prioritise eccentric parameter last
    :return par_ev_dict: orthogonal vectors
    """

    # Order list of parameter directions
    params = np.array(g.dx_directions.copy())
    final_params = []
    if eccentricity_last:
        final_params += ['ecc10', 'ecc10sqrd']
    if mass_ratio_last:
        final_params += ['mass_ratio', 'inverted_mass_ratio',
                         'symmetric_mass_ratio']
    for final_param in final_params:
        if final_param in params:
            params = np.delete(params, np.where(params == final_param))
            params = np.append(params, final_param)
    base_par_evs = np.identity(len(params))

    # Orthogonalise in dx space
    default_evs = [np.array(g.normalized_evecs()[param])
                   for param in params]
    dx_evs = np.matmul(np.linalg.inv(default_evs), base_par_evs)
    ev_list = dx_evs.T
    orth_ev_list = []
    for i in range(len(ev_list)):
        new_ev = ev_list[i].copy()
        for j in range(len(orth_ev_list)):
            norm = np.dot(orth_ev_list[j], orth_ev_list[j])
            new_ev -= np.dot(ev_list[i], orth_ev_list[j])*orth_ev_list[j]/norm
        orth_ev_list.append(new_ev)
    for i in range(len(orth_ev_list)):
        orth_ev_list[i] = orth_ev_list[i]/np.sqrt(np.dot(orth_ev_list[i],
                                                         orth_ev_list[i]))

    # Convert back to parameter space and set appropriate elements to zero
    # This avoids machine precision issues causing non-zero elements
    par_evs = np.matmul(default_evs, np.array(orth_ev_list).T)
    for i in range(len(par_evs)):
        for j in range(i+1, len(par_evs)):
            par_evs[:, i][j] = 0
    par_ev_dict = pe.SimplePESamples(SamplesDict(params, par_evs))

    return par_ev_dict

def calc_dx_bounds(
    current_point, dx_ind, g, orth_vectors, bound, maxs=None, mins=None
):
    """
    Calculates bounds along vector in dx space

    :param current_point: current_point in dx coordinates
    :param dx_ind: index of orth_vectors
    :param g: metric object
    :param orth_vectors: orthogonal vectors
    :param bound: maximum size of bounds
    :param maxs: dictionary of maximum values for physical parameters
    :param mins: dictionary of minimum values for physical parameters
    :return bounds: bounds
    """

    # Calculate unit change in physical parameters
    delta_dx = [0 for i in range(len(orth_vectors.keys()))]
    delta_dx[dx_ind] = 1
    delta_params_list = np.matmul(orth_vectors.samples, delta_dx)
    delta_params = pe.SimplePESamples(
        SamplesDict(orth_vectors.keys(),
                    [[param] for param in delta_params_list])
    )

    # Calculate current point in physical parameters
    base_point = current_point.copy()
    base_point[dx_ind] = 0
    current_delta_list = np.matmul(orth_vectors.samples, base_point)
    current_params = [[current_delta_list[i] + g.x[dx][0]]
                      for i, dx in enumerate(orth_vectors.keys())]
    current_params = pe.SimplePESamples(SamplesDict(orth_vectors.keys(),
                                                    current_params))

    # Calculate allowed bounds
    lower_alpha = waveform.check_physical(
        current_params, delta_params, -bound, maxs=maxs, mins=mins
    )
    chi_a_lower = current_params['chi_align'] - bound*lower_alpha*delta_params['chi_align']
    if 'chi_p' in current_params.keys():
        chi_p_lower = current_params['chi_p'] - bound*lower_alpha*delta_params['chi_p']
    else:
        chi_p_lower = np.sqrt(current_params['chi_p2'] - bound*lower_alpha*delta_params['chi_p2'])
    print(chi_a_lower, chi_p_lower, np.sqrt(chi_a_lower**2 + chi_p_lower**2))
    if lower_alpha < 1:
        lower_alpha = lower_alpha[0]
    upper_alpha = waveform.check_physical(
        current_params, delta_params, bound, maxs=maxs, mins=mins
    )
    chi_a_upper = current_params['chi_align'] + bound*upper_alpha*delta_params['chi_align']
    if 'chi_p' in current_params.keys():
        chi_p_upper = current_params['chi_p'] + bound*upper_alpha*delta_params['chi_p']
    else:
        chi_p_upper = np.sqrt(current_params['chi_p2'] + bound*upper_alpha*delta_params['chi_p2'])
    print(chi_a_upper, chi_p_upper, np.sqrt(chi_a_upper**2 + chi_p_upper**2))
    if upper_alpha < 1:
        upper_alpha = upper_alpha[0]
    bounds = [(-lower_alpha*bound, upper_alpha*bound)]

    return bounds

In [37]:
orth_vectors = create_metric_vectors(g)
bounds = calc_dx_bounds(
            [0,0,0,0], 3, g, orth_vectors, 100, maxs=None, mins=None
        )

[0.88741108] [0.43886397] [0.99]
[0.90329973] [0.001] [0.90330029]


# checking check_physical()

In [4]:
from simple_pe.waveforms import parameter_bounds, offset_params

def check_physical(x, dx, scaling, maxs=None, mins=None, verbose=False):
    """
    A function to check whether the point described by the positions x + dx is
    physically permitted.  If not, rescale and return the scaling factor

    :param x: dictionary with parameter values for initial point
    :param dx: dictionary with parameter variations
    :param scaling: the scaling to apply to dx
    :param maxs: a dictionary with the maximum permitted values of the
                 physical parameters
    :param mins: a dictionary with the minimum physical values of the physical
                 parameters
    :param verbose: print logging messages
    :return alpha: the scaling factor required to make x + scaling * dx
                   physically permissible
    """
    if mins is None:
        mins = parameter_bounds.param_mins

    if maxs is None:
        maxs = parameter_bounds.param_maxs
        maxs['a_1'] = 0.99999

    x0 = offset_params(x, dx, 0.)
    if verbose:
        print('initial point')
        print(x0)
    x_prime = offset_params(x, dx, scaling)
    if verbose:
        print('proposed point')
        print(x_prime)

    alpha = 1.

    if ('chi_p' in x_prime.keys()) or ('chi_p2' in x_prime.keys()):
        if ('chi_p2' in x_prime.keys()) and (x_prime['chi_p2'] <
                                             mins['chi_p2']):
            alpha = min(
                alpha, (x0['chi_p2'] - mins['chi_p2']) / (x0['chi_p2']
                                                          - x_prime['chi_p2'])
                                                          )
            x_prime['chi_p2'][0] = mins['chi_p2']
            if verbose:
                print("scaling to %.2f in direction %s" % (alpha, 'chi_p2'))
        x_prime.generate_spin_z()
        x_prime.generate_prec_spin()
        x0.generate_spin_z()
        x0.generate_prec_spin()

    for k, dx_val in x0.items():
        if k in mins.keys() and x_prime[k] < mins[k]:
            alpha = min(alpha, (x0[k] - mins[k]) / (x0[k] - x_prime[k]))
            if verbose:
                print("scaling to %.2f in direction %s" % (alpha, k))
        if k in maxs.keys() and x_prime[k] > maxs[k]:
            alpha = min(alpha, (maxs[k] - x0[k]) / (x_prime[k] - x0[k]))
            if verbose:
                print("scaling to %.2f in direction %s" % (alpha, k))

    # if varying 'chi_p2' need to double-check we don't go over limits
    if (
            ('chi_p2' in dx.keys() and (scaling * dx['chi_p2']))
            or ('chi_p' in dx.keys() and (scaling * dx['chi_p']))
        ):
        chia = "chi_eff" if "chi_eff" in x0.keys() else "chi_align"
        if 'chi_p' in x0.keys() and 'chi_p' in dx.keys():
            # need find alpha s.t.
            # (chi + alpha dchi)^2 + (chi_p + alpha dchi_p)^2 = max_spin^2
            c = x0[chia] ** 2 + x0['chi_p'] ** 2 - maxs['a_1']**2
            dcp = scaling * dx['chi_p']
            b = 2 * (
                x0[chia] * scaling * dx[chia] +
                x0['chi_p'] * dcp
            )
            a = (scaling * dx[chia]) ** 2 + dcp ** 2
        elif 'chi_p2' in x0.keys() and 'chi_p2' in dx.keys():
            # need find alpha s.t.
            # (chi + alpha dchi)^2 + chi_p2 + alpha dchi_p2 = max_spin^2
            c = x0[chia] ** 2 + x0['chi_p2'] - maxs['a_1']**2
            dcp2 = scaling * dx['chi_p2']
            b = 2 * x0[chia] * scaling * dx[chia] + dcp2
            a = (scaling * dx[chia]) ** 2
        else:
            raise ValueError("Cannot calculate precession scaling factor")
        if chia in dx.keys() and dx[chia]:
            alpha_prec = (-b + np.sqrt(b ** 2 - 4 * a * c)) / (2 * a)
        else:
            if 'chi_p2' in x0.keys() and 'chi_p2' in dx.keys():
                # not changing aligned spin, so easier
                # chi^2 + chi_p2 + alpha dchi_p2 = max_spin^2
                # if dchi_p2 < 0 then bound is positivity of chi_p2
                # else alpha = -c/dcp2
                if dcp2 < 0:
                    # chi_p2 + alpha dchi_p2 = mins['chi_p2']
                    alpha_prec = (mins['chi_p2'] - x0['chi_p2']) / dcp2
                else:
                    alpha_prec = -c / dcp2
            elif 'chi_p' in x0.keys() and 'chi_p' in dx.keys():
                # not changing aligned spin, so easier
                # chi^2 + (chi_p + alpha dchi_p)^2 = max_spin^2
                # if dchi_p < 0 then bound is positivity of chi_p
                # else solve the above equation for alpha
                if dcp < 0:
                    # chi_p + alpha dchi_p = mins['chi_p']
                    alpha_prec = (mins['chi_p'] - x0['chi_p']) / dcp
                else:
                    alpha_prec = (
                        np.sqrt(maxs['a_1']**2 - x0[chia]**2) - x0['chi_p']
                    ) / dcp
        if verbose:
            print("scaling to %.2f for precession" % alpha_prec)

        alpha = min(alpha, alpha_prec)

        if verbose:
            x_prime = offset_params(x, dx, alpha * scaling)
            print('new point')
            print(x_prime)

    return alpha


In [7]:
x = pe.SimplePESamples(SamplesDict(['chi_align', 'chi_p2', 'chirp_mass', 'symmetric_mass_ratio'], [[-0.407621], [0.826498], [35.683823], [0.188640]]))
dx = pe.SimplePESamples(SamplesDict(['chi_align', 'chi_p2', 'chirp_mass', 'symmetric_mass_ratio'], [[0.267538], [0.], [4.380127], [0.]]))
alpha = check_physical(x, dx, -2.631877341648345)
final_x = offset_params(x, dx, alpha*-2.631877341648345)
total_spin = np.sqrt(final_x['chi_align'][0]**2 + final_x['chi_p2'][0])
print(alpha)
print(final_x)
print(total_spin)

[0.82709346]
idx     chi_align      chi_p2         chirp_mass     symmetric_mass_ratio
0       -0.990000      0.826498       26.149125      0.188640       

1.344097466703959


In [59]:
x = pe.SimplePESamples(SamplesDict(['chi_align', 'chi_p2', 'chirp_mass', 'symmetric_mass_ratio'], [[0.2], [0.99000000001**2 - 0.2**2], [24], [0.2]]))
dx = pe.SimplePESamples(SamplesDict(['chi_align', 'chi_p2', 'chirp_mass', 'symmetric_mass_ratio'], [[-0.5], [0.5**2], [1], [0.01]]))
alpha = check_physical(x, dx, 1)
final_x = offset_params(x, dx, alpha)
total_spin = np.sqrt(final_x['chi_align'][0]**2 + final_x['chi_p2'][0])
print(alpha)
print(final_x)
print(total_spin)

[0.25] [0.05] [1.98000505e-11]
[0.16] [-3.9600101e-10]
[-3.9600101e-10]
idx     chi_align      chi_p2         chirp_mass     symmetric_mass_ratio
0       0.200000       0.940100       24.000000      0.200000       

0.99


In [70]:
i = 0
while True:
    random_vals = 2*np.random.rand(4) - 1
    random_vals[1] = np.abs(random_vals[1])**2
    random_vals[3] = np.sign(random_vals[3])*random_vals[3]**2
    random_chirp = 50*np.random.rand() + 10
    random_dchirp = 20*np.random.rand() - 10
    random_eta = 0.09*np.random.rand() + 0.1599
    random_deta = 0.2*np.random.rand() - 0.1
    if random_vals[0]**2 + random_vals[1] > 0.99**2:
        continue
    i += 1
    x = pe.SimplePESamples(SamplesDict(['chi_align', 'chi_p2', 'chirp_mass', 'symmetric_mass_ratio'], [[random_vals[0]], [random_vals[1]], [random_chirp], [random_eta]]))
    dx = pe.SimplePESamples(SamplesDict(['chi_align', 'chi_p2', 'chirp_mass', 'symmetric_mass_ratio'], [[random_vals[2]], [random_vals[3]], [random_dchirp], [random_deta]]))
    alpha = check_physical(x, dx, 1)
    final_x = offset_params(x, dx, alpha)
    total_spin = np.sqrt(final_x['chi_align'][0]**2 + final_x['chi_p2'][0])
    if total_spin > 0.9900001:
        print(random_vals)
        print(random_chirp, random_dchirp)
        print(random_eta, random_deta)
        print(alpha)
        print(final_x)
        print(total_spin)
        break
    if i%1000 == 0:
        print(f'tested {i}')

chi_p2 values smaller than the lower bound, smallest = 4.9e-07
setting equal to lower bound of 1e-06
chi_p2 values smaller than the lower bound, smallest = 1e-08
setting equal to lower bound of 1e-06
tested 1000
tested 2000
chi_p2 values smaller than the lower bound, smallest = 4.6e-08
setting equal to lower bound of 1e-06
chi_p2 values smaller than the lower bound, smallest = 3.1e-07
setting equal to lower bound of 1e-06
tested 3000
chi_p2 values smaller than the lower bound, smallest = 3e-07
setting equal to lower bound of 1e-06
chi_p2 values smaller than the lower bound, smallest = 7.3e-07
setting equal to lower bound of 1e-06
chi_p2 values smaller than the lower bound, smallest = 1.6e-10
setting equal to lower bound of 1e-06
tested 4000
chi_p2 values smaller than the lower bound, smallest = 2.5e-07
setting equal to lower bound of 1e-06
tested 5000
tested 6000
chi_p2 values smaller than the lower bound, smallest = 7.9e-07
setting equal to lower bound of 1e-06
tested 7000
chi_p2 valu

KeyboardInterrupt: 

# metric peak find test run

In [2]:
import time
import numpy as np
from simple_pe.param_est import metric, pe
from simple_pe.waveforms import (waveform_modes, waveform,
                                 parameter_bounds, eccentric)
from simple_pe import io
from scipy import optimize
from pesummary.utils.array import Array
from pesummary.utils.samples_dict import SamplesDict
from pesummary.gw.conversions.mass import (q_from_eta,
                                           component_masses_from_mchirp_q)
from simple_pe.param_est import filter as sp_filter

def find_peak_snr(
    ifos, data, psds, t_start, t_end, x, dx_directions, f_low,
    approximant="IMRPhenomD", method='scipy', scipy_method=None,
    scipy_opts=None, harm2=False, bounds=None, initial_mismatch=0.03,
    final_mismatch=0.001, tolerance=0.01, verbose=False, _net_snr=None,
    n_ecc_gen=6, iter_per_metric=3, large_shift_tol=1,
    metric_peak_bounds=3, ncpus=1, return_metrics=False
):
    """A function to find the maximum SNR.
    Either calculate a metric at the point x in dx_directions and walk to peak,
    or use scipy optimization regime

    :param ifos: list of ifos to use
    :param data: a dictionary of data from the given ifos
    :param psds: a dictionary of power spectra for the ifos
    :param t_start: start time to consider SNR peak
    :param t_end: end time to consider SNR peak
    :param x: dictionary with parameter values for initial point
    :param dx_directions: list of parameters for which to calculate waveform
    variations
    :param f_low: low frequency cutoff
    :param approximant: the approximant to use
    :param method: how to find the maximum (either 'scipy' or 'metric')
    :param scipy_method: scipy optimization method. Local methods: Nelder-Mead,
    L-BFGS-B, SLSQP (default), etc.
    :param scipy_opts: scipy optimization options (dict)
    :param harm2: use SNR from second harmonic
    :param bounds: give initial bounds for the range of parameters to
    investigate
    :param initial_mismatch: the mismatch for calculating the metric
    :param final_mismatch: the mismatch required to stop iteration
    :param tolerance: the allowed error in the metric is
    (tolerance * mismatch)
    :param verbose: if True then print info
    :param n_ecc_gen: number of component waveforms to generate for
                      eccentric harmonics
    :param iter_per_metric: number of iterations to perform per metric
    :param large_shift_tol: threshold for reducing metric mismatch
    :param metric_peak_bounds: bound size in dx coordinates
    :param ncpus: number of cpus to use for parallelisation
    :param return_metrics: whether to return the metrics calculated
    :return x_prime: the point in the grid with the highest snr
    :return snr_peak: the SNR squared at this point
    :return metrics: the metrics calculated (if return_metrics is True)
    """
    snr_peak = 0

    if method not in ["metric", "scipy", "grid", "particle_swarm"]:
        print(
            'Have only implemented metric, scipy, grid, particle '
            'swarm optimize based methods'
        )
        return

    elif method == "grid":
        if bounds is not None:
            mins = {dx: b[0] for dx, b in zip(dx_directions, bounds)}
            maxs = {dx: b[1] for dx, b in zip(dx_directions, bounds)}
        else:
            mins, maxs = None, None
        g_ms = metric.find_metric_and_eigendirections(
            x, dx_directions, _net_snr, f_low, psds["L1"], approximant,
            mins=mins, maxs=maxs
        )
        scale = 1.5
        npts = 11
        x_val = np.linspace(-scale, scale, npts)
        grid = np.meshgrid(x_val, x_val, x_val)
        n_evec = g_ms.normalized_evecs()
        grid_data = {}
        dx_data = np.tensordot(n_evec.samples, np.asarray(grid), axes=(1, 0))
        for i, d in enumerate(dx_directions):
            grid_data[d] = dx_data[i] + g_ms.x[d]
        snrs = filter_grid_components(
            x, grid_data, data, f_low, psds["L1"].sample_frequencies[-1],
            psds["L1"].delta_f, psds, approximant, t_start, t_end, ifos,
            prec_snr=True, hm_snr=True
        )
        # s = 'Total'
        amax = np.unravel_index(np.argmax(snrs), snrs.shape)
        snr_peak = snrs[amax]
        x = {}
        for k, i in grid_data.items():
            x[k] = Array([i[amax]])
        fixed_pars = {
            k: float(v) for k, v in x.items() if k not in dx_directions
        }
        x.update(fixed_pars)

    elif method == 'scipy':

        nlc = None
        if bounds is None:
            bounds = parameter_bounds.param_bounds(x, dx_directions, harm2)

        nlc = None
        # generate constraint on spins:
        chia = "chi_eff" if "chi_eff" in x.keys() else "chi_align"
        chip = "chi_p2" if "chi_p2" in x.keys() else "chi_p"
        if chip == "chi_p2":
            n = 1
        else:
            n = 2

        cond = (
            (chia in x) and (chip in x) and
            ((chia in dx_directions) or (chip in dx_directions))
        )
        if cond:
            # need bounds based on spin limits
            if (chia in dx_directions) and (chip in dx_directions):
                con = (
                    lambda y: y[dx_directions.index(chia)] ** 2 +
                    y[dx_directions.index(chip)] ** n
                )
                nlc = optimize.NonlinearConstraint(
                    con, parameter_bounds.param_mins['a_1']**2,
                    parameter_bounds.param_maxs['a_1']**2
                )

        x0 = np.array([x[k] for k in dx_directions]).flatten()
        fixed_pars = {
            k: float(v) for k, v in x.items() if k not in dx_directions
        }
        tested_methods = ['nelder-mead', 'l-bfgs-b', 'slsqp']
        if scipy_method and scipy_method.lower() not in tested_methods:
            print(
                f"Warning: Method '{scipy_method}' has not been tested. "
                "Proceed with caution and check results carefully."
            )

        # Check if method does not support constraints
        constraint_methods = ['cobyla', 'cobyqa', 'slsqp', 'trust-constr']

        if scipy_method and nlc is not None and (
            scipy_method.lower() not in constraint_methods
        ):
            print(
                f"Warning: Method '{scipy_method}' does not support "
                "constraints natively. Using penalty method to enforce "
                "constraints."
            )

            # Use penalized objective
            out = optimize.minimize(
                _penalized_objective, x0, args=(
                    nlc, dx_directions, ifos, data, psds, t_start, t_end,
                    f_low, approximant, fixed_pars, harm2, verbose
                ), bounds=bounds, method=scipy_method, options=scipy_opts
            )

        elif nlc is not None:
            # Method supports constraints - use them directly
            out = optimize.minimize(
                _neg_net_snr, x0, args=(
                    dx_directions, ifos, data, psds, t_start, t_end,
                    f_low, approximant, fixed_pars, harm2, verbose
                ), bounds=bounds, method=scipy_method, options=scipy_opts,
                constraints=nlc
            )
        else:
            # No constraints needed
            out = optimize.minimize(
                _neg_net_snr, x0, args=(
                    dx_directions, ifos, data, psds, t_start, t_end,
                    f_low, approximant, fixed_pars, harm2, verbose
                ), bounds=bounds, method=scipy_method, options=scipy_opts
            )

        x = {}
        for dx, val in zip(dx_directions, out.x):
            x[dx] = val
        x.update(fixed_pars)

        snr_peak = -out.fun

    elif method == 'metric':
        mismatch = initial_mismatch
        x = pe.SimplePESamples(x)
        fixed_pars = {
            k: float(v) for k, v in x.items() if k not in dx_directions
        }

        # Until reached specified mismatch
        finding_peak = True
        metric_list = []
        while finding_peak:

            # Reduce mismatch only when peak within 1dx with current mismatch
            large_shift = True
            while large_shift:

                # Create new metric and optimise along orthogonal vectors
                if 'ecc10' in dx_directions or 'ecc10sqrd' in dx_directions:
                    metric_approximant = f'{approximant}-Harms'
                else:
                    metric_approximant = approximant
                start = time.time()
                g = metric.find_metric_and_eigendirections(
                    x, dx_directions, snr=None, f_low=f_low,
                    psd=io.calculate_harmonic_mean_psd(psds),
                    approximant=metric_approximant, max_iter=1,
                    ncpus=ncpus, n_ecc_gen=n_ecc_gen,
                    mismatch=mismatch
                )
                metric_list.append(g)
                end = time.time()
                print(f'found metric in {end-start:.0f} seconds')
                start = time.time()
                peak_dxs, orth_vectors, snr_peak = _metric_find_peak(
                    ifos, data, psds, t_start, t_end, f_low, approximant,
                    g, bound=metric_peak_bounds, tolerance=tolerance,
                    n_ecc_gen=n_ecc_gen, n_iter=iter_per_metric,
                    fixed_pars=fixed_pars, harm2=harm2, verbose=verbose
                )
                end = time.time()

                # Calulate new point and measure distance moved
                delta_params_list = np.matmul(orth_vectors.samples, peak_dxs)
                params = [delta_params_list[i] + g.x[dx][0]
                          for i, dx in enumerate(orth_vectors.keys())]
                x = {x: params[i] for i, x in enumerate(orth_vectors.keys())}
                x.update(fixed_pars)
                print(f'found new point in {end-start:.0f} seconds:')
                for k, v in x.items():
                    print(f'{k}: {v:.5f}')
                shift = np.sqrt(np.dot(peak_dxs, peak_dxs))
                if shift < large_shift_tol:
                    large_shift = False
                else:
                    print(f'shift of {shift:.3f}dx with mismatch {mismatch} ' +
                          f'is larger than {large_shift_tol}dx, retaining ' +
                          'mismatch for next iteration')

            print(f"Found peak after shift of {shift:.3f}dx with mismatch " +
                  f"{mismatch}, reducing mismatch to refine")
            mismatch /= 4
            if mismatch < final_mismatch:
                finding_peak = False

    elif method == 'particle_swarm':
        import pyswarms as ps
        options = {
            'c1': 0.5,
            'c2': 3.0,
            'w': 0.01,
        }
        if bounds is None:
            bounds = []
            for dx in dx_directions:
                if dx == "chirp_mass":
                    bounds.append(
                        (x['chirp_mass'][0] * 0.75, x['chirp_mass'][0] * 1.33)
                    )
                elif dx == "symmetric_mass_ratio":
                    bounds.append((0.1, 0.249999))
                elif dx == "chi_align" or dx == "spin_1z" or dx == "spin_2z":
                    bounds.append((-0.99, 0.99))
                elif dx == "chi_p" or dx == "chi_p2":
                    bounds.append(0., 0.99)

        min_bounds = np.array([_[0] for _ in bounds])
        max_bounds = np.array([_[1] for _ in bounds])

        # Initialize the population with random values within specified bounds
        population = np.random.uniform(
            min_bounds, max_bounds, size=(500, len(bounds))
        )

        # add the initial point to the population
        population = np.concatenate(
            (population[:-1], np.atleast_2d([x[k] for k in dx_directions]).T)
        )

        optimizer = ps.single.GlobalBestPSO(
            n_particles=500,
            dimensions=len(dx_directions),
            options=options,
            bounds=(min_bounds, max_bounds),
            init_pos=population
        )
        # Generate constraint on spins (chi_align^2 + chi_p^n < 1)
        constraint_info = None
        chia = "chi_eff" if "chi_eff" in x.keys() else "chi_align"
        chip = "chi_p2" if "chi_p2" in x.keys() else "chi_p"
        if chip == "chi_p2":
            n = 1
        else:
            n = 2

        cond = (
            (chia in x) and (chip in x) and
            ((chia in dx_directions) or (chip in dx_directions))
        )
        if cond:
            # need bounds based on spin limits
            if (chia in dx_directions) and (chip in dx_directions):
                # Store constraint info in a picklable dict
                lb = parameter_bounds.param_mins['a_1'] ** 2
                ub = parameter_bounds.param_maxs['a_1'] ** 2
                constraint_info = {
                    'chia_idx': dx_directions.index(chia),
                    'chip_idx': dx_directions.index(chip),
                    'n': n,
                    'lb': lb,
                    'ub': ub
                }
                print(
                    f"Applying constraint: {chia}^2 + {chip}^{n} < {ub} "
                    f"for particle swarm optimization"
                )

        extra_args = [
            dx_directions, constraint_info, ifos, data, psds, t_start,
            t_end, f_low, approximant
        ]
        fixed_pars = {
            k: float(v) for k, v in x.items() if k not in dx_directions
        }

        snr_peak, x = optimizer.optimize(
            _neg_net_snr_pso,
            iters=8,
            n_processes=1,
            args=extra_args
        )
        x = {key: x[num] for num, key in enumerate(dx_directions)}
        x.update(fixed_pars)
        snr_peak *= -1
    if return_metrics:
        return x, snr_peak, metric_list
    else:
        return x, snr_peak


def create_metric_vectors(
    g, mass_ratio_last=True, eccentricity_last=True
):
    """
    Calculates orthogonal vectors of metric dx space.

    :param g: metric object
    :param mass_ratio_last: whether to prioritise mass ratio parameter last
    :param eccentricity_last: whether to prioritise eccentric parameter last
    :return par_ev_dict: orthogonal vectors
    """

    # Order list of parameter directions
    params = np.array(g.dx_directions.copy())
    final_params = []
    if eccentricity_last:
        final_params += ['ecc10', 'ecc10sqrd']
    if mass_ratio_last:
        final_params += ['mass_ratio', 'inverted_mass_ratio',
                         'symmetric_mass_ratio']
    for final_param in final_params:
        if final_param in params:
            params = np.delete(params, np.where(params == final_param))
            params = np.append(params, final_param)
    base_par_evs = np.identity(len(params))

    # Orthogonalise in dx space
    default_evs = [np.array(g.normalized_evecs()[param])
                   for param in params]
    dx_evs = np.matmul(np.linalg.inv(default_evs), base_par_evs)
    ev_list = dx_evs.T
    orth_ev_list = []
    for i in range(len(ev_list)):
        new_ev = ev_list[i].copy()
        for j in range(len(orth_ev_list)):
            norm = np.dot(orth_ev_list[j], orth_ev_list[j])
            new_ev -= np.dot(ev_list[i], orth_ev_list[j])*orth_ev_list[j]/norm
        orth_ev_list.append(new_ev)
    for i in range(len(orth_ev_list)):
        orth_ev_list[i] = orth_ev_list[i]/np.sqrt(np.dot(orth_ev_list[i],
                                                         orth_ev_list[i]))

    # Convert back to parameter space and set appropriate elements to zero
    # This avoids machine precision issues causing non-zero elements
    par_evs = np.matmul(default_evs, np.array(orth_ev_list).T)
    for i in range(len(par_evs)):
        for j in range(i+1, len(par_evs)):
            par_evs[:, i][j] = 0
    par_ev_dict = pe.SimplePESamples(SamplesDict(params, par_evs))

    return par_ev_dict


def calc_dx_bounds(
    current_point, dx_ind, g, orth_vectors, bound, maxs=None, mins=None
):
    """
    Calculates bounds along vector in dx space

    :param current_point: current_point in dx coordinates
    :param dx_ind: index of orth_vectors
    :param g: metric object
    :param orth_vectors: orthogonal vectors
    :param bound: maximum size of bounds
    :param maxs: dictionary of maximum values for physical parameters
    :param mins: dictionary of minimum values for physical parameters
    :return bounds: bounds
    """

    # Calculate unit change in physical parameters
    delta_dx = [0 for i in range(len(orth_vectors.keys()))]
    delta_dx[dx_ind] = 1
    delta_params_list = np.matmul(orth_vectors.samples, delta_dx)
    delta_params = pe.SimplePESamples(
        SamplesDict(orth_vectors.keys(),
                    [[param] for param in delta_params_list])
    )

    # Calculate current point in physical parameters
    base_point = current_point.copy()
    base_point[dx_ind] = 0
    current_delta_list = np.matmul(orth_vectors.samples, base_point)
    current_params = [[current_delta_list[i] + g.x[dx][0]]
                      for i, dx in enumerate(orth_vectors.keys())]
    current_params = pe.SimplePESamples(SamplesDict(orth_vectors.keys(),
                                                    current_params))

    # Calculate allowed bounds
    lower_alpha = waveform.check_physical(
        current_params, delta_params, -bound, maxs=maxs, mins=mins
    )
    if lower_alpha < 1:
        lower_alpha = lower_alpha[0]
    upper_alpha = waveform.check_physical(
        current_params, delta_params, bound, maxs=maxs, mins=mins
    )
    if upper_alpha < 1:
        upper_alpha = upper_alpha[0]
    bounds = [(-lower_alpha*bound, upper_alpha*bound)]

    return bounds


def _metric_find_peak(
    ifos, data, psds, t_start, t_end, f_low, approximant, g,
    bound, tolerance, n_ecc_gen=6, n_iter=1,
    maxs=None, mins=None, fixed_pars=None, harm2=False, verbose=False
):
    """
    Finds peak by sequentially optimising over orthogonal vectors of metric.

    :param ifos: list of ifos to use
    :param data: a dictionary of data from the given ifos
    :param psds: a dictionary of power spectra for the ifos
    :param t_start: start time to consider SNR peak
    :param t_end: end time to consider SNR peak
    :param f_low: low frequency cutoff
    :param approximant: the approximant to use
    :param g: metric object
    :param bound: maximum size of bounds in dx coordinates
    :param tolerance: tolerance of optimisations in dx coordinates
    :param n_ecc_gen: number of component waveforms to generate
                      eccentric harmonics
    :param n_iter: number of times to repeat optimisation
    :param maxs: dictionary of maximum values for physical parameters
    :param mins: dictionary of minimum values for physical parameters
    :param fixed_pars: a dictionary of fixed parameters and values
    :param harm2: if True then generate two harmonics and filter both
    :param verbose: if True then print info
    :return current_point: peak point found in dx coordinates
    :return orth_vectors: orthogonal vectors that define dx space
    :return -snr: the negative of the match at this point
    """

    # Calculate orthogonal vectors
    orth_vectors = create_metric_vectors(g)

    # Start optimisation
    current_point = np.zeros(len(orth_vectors.keys()))
    harm_psd = io.calculate_harmonic_mean_psd(psds)
    for big_i in range(n_iter):
        ind_order = []
        iterating_inds = np.arange(len(orth_vectors.keys()))
        while len(iterating_inds) > 0:
            start_iter_point = current_point.copy()
            start_iter_inds = iterating_inds.copy()

            # Calculate eccentric harmonics and target mean anomaly if needed
            if (
                'ecc10' in orth_vectors.keys() or
                'ecc10sqrd' in orth_vectors.keys()
            ):
                delta_params_list = np.matmul(orth_vectors.samples,
                                              current_point)
                params = [delta_params_list[i] + g.x[dx][0]
                          for i, dx in enumerate(orth_vectors.keys())]
                s = dict(zip(orth_vectors.keys(), params))
                if fixed_pars is not None:
                    s.update(fixed_pars)
                ecc_harm_psds = psds.copy()
                ecc_harm_psds['harm'] = harm_psd
                ecc_harms = waveform.make_waveform(
                    s, psds[ifos[0]].delta_f, f_low, len(psds[ifos[0]]),
                    f'{approximant}-Harms', n_ecc_gen=n_ecc_gen,
                    n_ecc_harms=3, ecc_harm_psd=ecc_harm_psds
                )
                z_ecc = {}
                modes = [0, 1, -1]
                for ifo in ifos:
                    z_ecc[ifo], _ = waveform_modes.calculate_mode_snr(
                        data[ifo], psds[ifo], ecc_harms[ifo], t_start, t_end,
                        f_low, modes, subsample_interpolation=True,
                        dominant_mode=0
                    )
                _, mode_snrs = waveform_modes.network_mode_snr(
                    z_ecc, ifos, modes, dominant_mode=0, return_cplx=True
                )
                _, target_ma = waveform_modes.two_ecc_harms_snr(
                    {k: np.abs(mode_snrs[k]) for k in [0, 1, -1]},
                    {k: np.angle(mode_snrs[k]) for k in [0, 1, -1]}
                )
            else:
                ecc_harms = None
                target_ma = None

            # Optimise over each eigenvector
            working_inds = iterating_inds.copy()
            for i in range(len(iterating_inds)):

                # Decide optimisation order on first iteration
                if len(ind_order) < len(orth_vectors.keys()):
                    # Calculate bounds for each eigenvector
                    bounds_list = []
                    bounds_size = []
                    for j in working_inds:
                        bounds = calc_dx_bounds(
                            current_point, j, g, orth_vectors, bound,
                            maxs=maxs, mins=mins
                        )
                        bounds_list.append(bounds)
                        bounds_size.append(np.abs(np.prod(bounds)))

                    # Choose lowest range of bounds for next eigenvector
                    if verbose:
                        print(bounds_list)
                        print(bounds_size)
                    next_ind = np.argmax(bounds_size)
                    next_ev = working_inds[next_ind]
                    next_bounds = bounds_list[next_ind]
                    ind_order.append(next_ev)

                # Follow ind_order otherwise
                else:
                    for ind in ind_order:
                        if ind in working_inds:
                            next_ev = ind
                            next_bounds = calc_dx_bounds(
                                current_point, ind, g, orth_vectors,
                                bound, maxs=maxs, mins=mins
                            )
                            break

                working_inds = np.delete(working_inds,
                                         np.where(working_inds == next_ev))
                if verbose:
                    print(f'optimising over dx ind {next_ev}')
                    print(f'with bounds {next_bounds}')

                # Perform optimisation
                out = optimize.minimize(
                    sp_filter._neg_net_snr_metric, current_point[next_ev], args=(
                        next_ev, current_point, ifos, data, psds, t_start,
                        t_end, f_low, approximant, g, orth_vectors, ecc_harms,
                        target_ma, harm_psd, maxs, mins, fixed_pars, harm2,
                        verbose
                    ), bounds=next_bounds, method='Powell',
                    options={'xtol': tolerance, 'ftol': np.inf}
                )
                current_point[next_ev] = out.x[0]
                if verbose:
                    print(f'new peak point: {current_point}')

                # Check if railing in this direction
                lower_gap = np.abs(np.diff([out.x[0], next_bounds[0][0]]))
                not_lower_rail = lower_gap > 3*tolerance
                upper_gap = np.abs(np.diff([out.x[0], next_bounds[0][1]]))
                not_upper_rail = upper_gap > 3*tolerance
                if not_lower_rail and not_upper_rail:
                    del_inds = np.where(iterating_inds == next_ev)
                    iterating_inds = np.delete(iterating_inds,
                                               del_inds)
                elif verbose:
                    print(f'dx ind {next_ev} is railing')

            # Stop if only direction left was just iterated over
            if len(iterating_inds) == 1 and iterating_inds[0] == next_ev:
                iterating_inds = []
            # Stop if point did not significantly change in last iteration
            elif len(iterating_inds) > 0:
                iter_change = current_point - start_iter_point
                iter_change_dist = np.sqrt(np.dot(iter_change, iter_change))
                if iter_change_dist < np.sqrt(len(start_iter_inds))*tolerance:
                    iterating_inds = []

        if verbose:
            print(f'after iteration {big_i}: {current_point}')

    return current_point, orth_vectors, -out.fun

In [3]:
from simple_pe import io
import json
f_low = 15
asd_data = {'H1': '/home/mukesh.singh/work/Simple-PE/benchmarking/O4_psds/aligo_O4high_PSD.txt',
            'L1': '/home/mukesh.singh/work/Simple-PE/benchmarking/O4_psds/aligo_O4high_PSD.txt',
            'V1': '/home/mukesh.singh/work/Simple-PE/benchmarking/O4_psds/avirgo_O4high_NEW_PSD.txt'}
psds = io.load_psd_from_file(
           {}, asd_data, 1/32, 15, 2048,
       )
hm_psd = io.calculate_harmonic_mean_psd(psds)

data_dict = {'H1': '/home/mukesh.singh/work/Simple-PE/benchmarking/precessing_random/simple_pe_prior_injections/gaussian_noise_metric_peak_finder_spin_bound/injection_12/outdir/output/H1-INJECTION.gwf',
             'L1': '/home/mukesh.singh/work/Simple-PE/benchmarking/precessing_random/simple_pe_prior_injections/gaussian_noise_metric_peak_finder_spin_bound/injection_12/outdir/output/L1-INJECTION.gwf',
             'V1': '/home/mukesh.singh/work/Simple-PE/benchmarking/precessing_random/simple_pe_prior_injections/gaussian_noise_metric_peak_finder_spin_bound/injection_12/outdir/output/V1-INJECTION.gwf'}
channel_dict = {'H1': 'HWINJ_INJECTED',
                'L1': 'HWINJ_INJECTED',
                'V1': 'HWINJ_INJECTED'}
trigger_parameters = io.load_trigger_parameters_from_file(
        '/home/mukesh.singh/work/Simple-PE/benchmarking/precessing_random/simple_pe_prior_injections/gaussian_noise_metric_peak_finder_spin_bound/injection_12/trigger_parameters.json', 'IMRPhenomXPHM'
    )
strain, strain_f = io.load_strain_data_from_file(
    trigger_parameters, data_dict, channel_dict, 15,
    2046, minimum_data_length=32
)

/home/ben.patterson/.conda/envs/igwn_eccentric_new/lib/python3.10/site-packages/pycbc/types/array.py:390: RuntimeWarning: divide by zero encountered in divide
  return self._data.__rtruediv__(other)
2026-07-22  15:55:03 PESummary WARNING : Could not find f_final in input file and one was not passed from the command line. Using 1024.0Hz as default
2026-07-22  15:55:03 PESummary WARNING : Could not find delta_f in input file and one was not passed from the command line. Using 0.00390625Hz as default
2026-07-22  15:55:03 PESummary WARNING : Could not find f_start in input file and one was not passed from the command line. Using 20.0Hz as default
2026-07-22  15:55:03 PESummary WARNING : Could not find f_low in input file and one was not passed from the command line. Using 20.0Hz as default
2026-07-22  15:55:03 PESummary WARNING : Could not find reference_frequency in input file. Using 20Hz as default
2026-07-22  15:55:03 PESummary INFO    : Averaging the final spin from the following fits:

In [4]:
x = {'chirp_mass': 56.0183, 'symmetric_mass_ratio': 0.1759, 'chi_align': -0.5450, 'chi_p2': 0.1984}
find_peak_snr(
    ['H1', 'L1', 'V1'], strain_f, psds, -0.1, 0.1, x, list(x.keys()), 15,
    approximant="IMRPhenomXPHM", method='metric', scipy_method=None,
    scipy_opts=None, harm2=True, bounds=None, initial_mismatch=0.03,
    final_mismatch=0.001, tolerance=0.01, verbose=True, _net_snr=None,
    n_ecc_gen=6, iter_per_metric=3, large_shift_tol=1,
    metric_peak_bounds=3, ncpus=1, return_metrics=False
)

2026-07-22  15:55:03 PESummary WARNING : Could not find f_final in input file and one was not passed from the command line. Using 1024.0Hz as default
2026-07-22  15:55:03 PESummary WARNING : Could not find f_start in input file and one was not passed from the command line. Using 20.0Hz as default
Calculating the metric | iteration 0 < 1| error 0.027 > 0.0015

chi_p2 values smaller than the lower bound, smallest = 1e-06
setting equal to lower bound of 1e-06
chi_p2 values smaller than the lower bound, smallest = 1e-06
setting equal to lower bound of 1e-06


Failed to achieve requested tolerance.  Requested: 0.0015achieved 0.06: 


found metric in 59 seconds
[[(-3.0, 3.0)], [(-0.6178639792132516, 2.1312836136906554)], [(-0.42298248170896774, 0.7222302620622776)], [(-1.5731262067979448, 0.4167469817877901)]]
[9.0, 1.316843374386907, 0.3054907486124203, 0.6555955986543185]
optimising over dx ind 0
with bounds [(-3.0, 3.0)]
trying dx coordinate of [0.]
making waveform at parameters
{'chirp_mass': 56.0183, 'chi_align': -0.545, 'chi_p2': 0.1984, 'symmetric_mass_ratio': 0.1759}
32.0
32.0


ValueError: Length of template and data must match

# zero length dx scaling

Now investigating case where a dx can be scaled to zero causing a singular matrix.

In [2]:
filename = '/home/ben.patterson/projects/ecc_simple_pe_runs/random_sets/gaussian_noise_50_v16/failed_tests/injection_41/scale_dx_inputs.pkl'
with open(filename, 'rb') as handle:
    scale_dx_inputs = pickle.load(handle)

In [3]:
from simple_pe.param_est.metric import *
import logging
from scipy import optimize

def scale_dx(x, dx, desired_mismatch, f_low, psd, h0=None,
             approximant="IMRPhenomD", tolerance=1e-2, mins=None, maxs=None,
             n_ecc_gen=6):

    _logger = logging.getLogger('PESummary')
    _logger.setLevel(logging.CRITICAL + 10)
    for num in range(10):
        try:
            opt = optimize.root_scalar(
                lambda a: average_mismatch(
                    x, dx, a, f_low, psd, h0=h0, approximant=approximant,
                    mins=mins, maxs=maxs, n_ecc_gen=n_ecc_gen
                ) - desired_mismatch,
                bracket=np.array([0., 20.]) * (float(num) + 1),
                method='brentq', rtol=tolerance
            )
            break
        except ValueError:
            continue

    try:
        scale = np.max([opt.root, 1e-10])
    except UnboundLocalError:
        raise ValueError("Unable to scale the input vectors")
    return scale

In [6]:
from simple_pe.param_est.metric import scale_dx
scale_dx(*scale_dx_inputs)

0.04814022434884348

In [5]:
scale_dx_inputs

[{'ecc10sqrd': Array([0.]),
  'chirp_mass': Array([39.12326558]),
  'symmetric_mass_ratio': Array([0.17195467]),
  'chi_align': Array([-0.99]),
  'distance': Array([1.])},
 {'ecc10sqrd': Array([-0.70710678]),
  'chirp_mass': Array([-2.71244418e-34]),
  'symmetric_mass_ratio': Array([1.22039207e-32]),
  'chi_align': Array([-0.70710678])},
 0.01,
 20.0,
 'TEOBResumS-Dali-Harms',
 0.05,
 None,
 None,
 6]